# Week 6 Day 5: Fine-tuning a Frontier Model

Community contribution — fine-tuning project following the steps in **Week 6 Day 5**.

We use OpenAI's API to fine-tune a private variant of **GPT-4.1-nano** for the product pricer task.

## Setup: imports and paths

Add the week6 root to the path so we can import from `pricer`. Run this notebook from `week6/community-contributions/emmanuel_ochade/`.

In [ ]:
import os
import sys
import re
import json
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

# Ensure week6 is on the path when running from community-contributions/emmanuel_ochade
week6_root = Path.cwd().parent.parent
if str(week6_root) not in sys.path:
    sys.path.insert(0, str(week6_root))

from pricer.items import Item
from pricer.evaluator import evaluate

## Environment

## Config

Tweak these to run different experiments (model, suffix, data size, epochs).

In [ ]:
# Fine-tuning and data sizes (easy to tweak)
FINE_TUNE_MODEL = "gpt-4.1-nano-2025-04-14"
FINE_TUNE_SUFFIX = "pricer"
N_EPOCHS = 1
BATCH_SIZE = 1
TRAIN_SIZE = 100
VAL_SIZE = 50

In [ ]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"  # I dont have mine yet
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
openai_client = OpenAI()

## Data size

OpenAI recommends fine-tuning with 50–100 examples. We use 100 training and 50 validation examples (1 epoch) to keep cost minimal.

In [ ]:
fine_tune_train = train[:TRAIN_SIZE]
fine_tune_validation = val[:VAL_SIZE]

len(fine_tune_train)

## Step 1: Prepare data in JSONL format and upload to OpenAI

Each line is a JSON object with a `messages` array: user message (product summary) and assistant message (price).

In [ ]:
# Single prompt template for training and inference (avoids drift)
PROMPT_TEMPLATE = "Estimate the price of this product. Respond with the price, no explanation\n\n{text}"

def messages_for(item):
    text = item.summary or item.title or ""
    message = PROMPT_TEMPLATE.format(text=text)
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

In [ ]:
# PROMPT_TEMPLATE and messages_for() defined in the cell above.

In [ ]:
messages_for(fine_tune_train[0])

In [ ]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + "}\n"
    return result.strip()

In [ ]:
print(make_jsonl(train[:3]))

In [ ]:
def write_jsonl(items, filename):
    jsonl_dir = Path.cwd() / "jsonl"
    jsonl_dir.mkdir(exist_ok=True)
    path = jsonl_dir / filename
    with open(path, "w") as f:
        f.write(make_jsonl(items))

In [ ]:
write_jsonl(fine_tune_train, "fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "fine_tune_validation.jsonl")

In [ ]:
jsonl_dir = Path.cwd() / "jsonl"
with open(jsonl_dir / "fine_tune_train.jsonl", "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
train_file

In [ ]:
with open(jsonl_dir / "fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai_client.files.create(file=f, purpose="fine-tune")
validation_file

## Step 2: Create the fine-tuning job

In [ ]:
job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model=FINE_TUNE_MODEL,
    seed=42,
    hyperparameters={"n_epochs": N_EPOCHS, "batch_size": BATCH_SIZE},
    suffix=FINE_TUNE_SUFFIX,
)
job_id = job.id
print(f"Job ID: {job_id}")

In [ ]:
openai_client.fine_tuning.jobs.list(limit=1)

In [ ]:
openai_client.fine_tuning.jobs.retrieve(job_id)

In [ ]:
openai_client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

### Wait for the job to complete (optional)

Poll until status is `succeeded` or `failed`. Skip this if you already have a finished job and will set `job_id` or `fine_tuned_model_name` manually.

In [ ]:
import time

def wait_for_job(job_id, poll_interval=30, max_wait=3600):
    """Poll until job is succeeded or failed. Returns the job object."""
    start = time.time()
    while time.time() - start < max_wait:
        job = openai_client.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        print(f"  Status: {status}")
        if status == "succeeded":
            print(f"  Fine-tuned model: {job.fine_tuned_model}")
            return job
        if status == "failed":
            print(f"  Job failed: {getattr(job, 'error', job)}")
            return job
        time.sleep(poll_interval)
    raise TimeoutError("Job did not complete in time")

# Uncomment to wait for the current job_id:
# wait_for_job(job_id)

## Step 3: Test the fine-tuned model

Once the job status is `succeeded`, retrieve the model name and run inference and evaluation.

In [ ]:
fine_tuned_model_name = openai_client.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
fine_tuned_model_name

In [ ]:
def test_messages_for(item):
    text = item.summary or item.title or ""
    message = PROMPT_TEMPLATE.format(text=text)
    return [{"role": "user", "content": message}]

In [ ]:
test_messages_for(test[0])

In [ ]:
def gpt_4_1_nano_fine_tuned(item):
    response = openai_client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7,
    )
    return response.choices[0].message.content

## Summary

- **Job ID:** use the `job_id` from Step 2.
- **Model:** `fine_tuned_model_name` (set in Step 3 once the job has succeeded).
- **Test score:** the `evaluate(...)` cell above prints the average absolute error on the test set (lower is better; course target is around $76).

In [ ]:
# Quick reference (run after fine-tuned model is ready)
print(f"Job ID:          {job_id}")
print(f"Fine-tuned model: {fine_tuned_model_name}")
print("Run evaluate(gpt_4_1_nano_fine_tuned, test) above for the test set score.")

In [ ]:
print(test[0].price)
print(gpt_4_1_nano_fine_tuned(test[0]))

In [ ]:
evaluate(gpt_4_1_nano_fine_tuned, test)